#### `train_comportemental.ipynb`

#### Cellule 1 : Description (markdown)

### Entraînement du modèle XGBoost comportemental
Fusion Teen + Adults, SMOTE, GridSearch, évaluation, sauvegarde joblib.

### Cellule 2 : Importations et configuration (code)

In [7]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Entraînement du modèle XGBoost comportemental\n",
    "Fusion Teen + Adults, SMOTE, GridSearch, évaluation, sauvegarde joblib."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import logging\n",
    "import sys\n",
    "from pathlib import Path\n",
    "import numpy as np\n",
    "import pandas as pd\n",
    "from imblearn.over_sampling import SMOTE\n",
    "from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score\n",
    "from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split\n",
    "from sklearn.preprocessing import LabelEncoder, StandardScaler\n",
    "from xgboost import XGBClassifier\n",
    "\n",
    "# ---- CORRECTION POUR LE NOTEBOOK ----\n",
    "# Dans un notebook, __file__ n'existe pas. On remonte depuis le répertoire courant.\n",
    "# Supposons que ce notebook se trouve dans le dossier 'models/comportemental/'.\n",
    "# Alors la racine du projet est deux niveaux au-dessus.\n",
    "root_dir = Path.cwd().parent.parent\n",
    "sys.path.insert(0, str(root_dir))\n",
    "# ------------------------------------\n",
    "\n",
    "from models.comportemental.features import UNIFIED_FEATURES, build_combined_dataset\n",
    "from utils.config import RANDOM_STATE, SMOTE_STRATEGY, TEST_SIZE, XGBOOST_PARAMS\n",
    "from utils.model_loader import save_comportemental\n",
    "\n",
    "logging.basicConfig(level=logging.INFO, format=\"%(asctime)s [%(levelname)s] %(name)s — %(message)s\")\n",
    "logger = logging.getLogger(__name__)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Étape 1 – Chargement et fusion des données"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def load_and_prepare_data():\n",
    "    logger.info(\"=== CHARGEMENT DES DONNÉES ===\")\n",
    "    df = build_combined_dataset()\n",
    "    feature_cols = [f for f in UNIFIED_FEATURES if f in df.columns]\n",
    "    for trend_col in [\"stress_trend\", \"sleep_trend\"]:\n",
    "        if trend_col in df.columns:\n",
    "            feature_cols.append(trend_col)\n",
    "    X = df[feature_cols].copy()\n",
    "    y = df[\"label\"].values.astype(int)\n",
    "    logger.info(f\"Features retenues : {feature_cols}\\nDistribution cible : {dict(zip(*np.unique(y, return_counts=True)))}\")\n",
    "    return X, y, feature_cols\n",
    "\n",
    "X, y, feature_names = load_and_prepare_data()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Étape 2 – Prétraitement (normalisation)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def preprocess(X_train, X_test):\n",
    "    scaler = StandardScaler()\n",
    "    X_train_scaled = scaler.fit_transform(X_train)\n",
    "    X_test_scaled = scaler.transform(X_test)\n",
    "    return X_train_scaled, X_test_scaled, scaler"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Étape 3 – Split train / validation / test"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "X_train_raw, X_test_raw, y_train, y_test = train_test_split(\n",
    "    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y\n",
    ")\n",
    "X_train_raw, X_val_raw, y_train, y_val = train_test_split(\n",
    "    X_train_raw, y_train, test_size=0.15, random_state=RANDOM_STATE, stratify=y_train\n",
    ")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Étape 4 – Normalisation"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "X_train_scaled, X_test_scaled, scaler = preprocess(X_train_raw, X_test_raw)\n",
    "_, X_val_scaled, _ = preprocess(X_train_raw, X_val_raw)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Étape 5 – SMOTE sur l’entraînement"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def apply_smote(X_train, y_train):\n",
    "    logger.info(\"Application de SMOTE…\")\n",
    "    logger.info(f\"Avant SMOTE — Distribution : {dict(zip(*np.unique(y_train, return_counts=True)))}\")\n",
    "    smote = SMOTE(sampling_strategy=SMOTE_STRATEGY, random_state=RANDOM_STATE)\n",
    "    X_res, y_res = smote.fit_resample(X_train, y_train)\n",
    "    logger.info(f\"Après SMOTE — Distribution : {dict(zip(*np.unique(y_res, return_counts=True)))}\")\n",
    "    return X_res, y_res\n",
    "\n",
    "X_train_res, y_train_res = apply_smote(X_train_scaled, y_train)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Étape 6 – Entraînement XGBoost avec early stopping"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def train_xgboost(X_train, y_train, X_val, y_val):\n",
    "    logger.info(\"=== ENTRAÎNEMENT XGBOOST ===\")\n",
    "    params = XGBOOST_PARAMS.copy()\n",
    "    n_estimators = params.pop(\"n_estimators\")\n",
    "    model = XGBClassifier(**params, n_estimators=n_estimators, early_stopping_rounds=30)\n",
    "    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=50)\n",
    "    logger.info(f\"Meilleur n_estimators : {model.best_iteration}\")\n",
    "    return model\n",
    "\n",
    "model = train_xgboost(X_train_res, y_train_res, X_val_scaled, y_val)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Étape 7 – Évaluation sur le jeu de test"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "y_pred = model.predict(X_test_scaled)\n",
    "y_proba = model.predict_proba(X_test_scaled)[:, 1]\n",
    "auc = roc_auc_score(y_test, y_proba)\n",
    "report = classification_report(y_test, y_pred, target_names=[\"Sain\", \"À risque\"])\n",
    "cm = confusion_matrix(y_test, y_pred)\n",
    "\n",
    "logger.info(f\"AUC-ROC : {auc:.4f}\")\n",
    "logger.info(f\"Rapport de classification :\\n{report}\")\n",
    "logger.info(f\"Matrice de confusion :\\n{cm}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Visualisations (courbe ROC, matrice de confusion)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "from sklearn.metrics import roc_curve\n",
    "\n",
    "# Dossier de sortie\n",
    "OUTPUT_DIR = Path.cwd().parent.parent / \"outputs\" / \"comportemental\"\n",
    "OUTPUT_DIR.mkdir(parents=True, exist_ok=True)\n",
    "\n",
    "# Courbe ROC\n",
    "fpr, tpr, _ = roc_curve(y_test, y_proba)\n",
    "plt.figure()\n",
    "plt.plot(fpr, tpr, label=f'AUC = {auc:.3f}')\n",
    "plt.plot([0, 1], [0, 1], 'k--')\n",
    "plt.xlabel('False Positive Rate')\n",
    "plt.ylabel('True Positive Rate')\n",
    "plt.title('Courbe ROC - Modèle comportemental')\n",
    "plt.legend(loc='lower right')\n",
    "plt.savefig(OUTPUT_DIR / 'roc_curve_comportemental.png', dpi=150)\n",
    "plt.show()\n",
    "plt.close()\n",
    "\n",
    "# Matrice de confusion\n",
    "plt.figure(figsize=(6,4))\n",
    "sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Sain','À risque'], yticklabels=['Sain','À risque'])\n",
    "plt.title('Matrice de confusion - Modèle comportemental')\n",
    "plt.ylabel('Vraie classe')\n",
    "plt.xlabel('Prédiction')\n",
    "plt.savefig(OUTPUT_DIR / 'confusion_matrix_comportemental.png', dpi=150, bbox_inches='tight')\n",
    "plt.show()\n",
    "plt.close()\n",
    "\n",
    "# Sauvegarde CSV de la matrice\n",
    "cm_df = pd.DataFrame(cm, index=['Sain','À risque'], columns=['Prédit Sain','Prédit À risque'])\n",
    "cm_df.to_csv(OUTPUT_DIR / 'confusion_matrix_comportemental.csv')\n",
    "print(\"Graphiques sauvegardés dans\", OUTPUT_DIR)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Validation croisée (optionnelle)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "model_cv = XGBClassifier(\n",
    "    n_estimators=100, max_depth=6, learning_rate=0.05,\n",
    "    subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_STATE, n_jobs=-1\n",
    ")\n",
    "cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)\n",
    "cv_scores = cross_val_score(model_cv, X_test_scaled, y_test, cv=cv, scoring='roc_auc')\n",
    "logger.info(f\"CV AUC (5-fold) : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Sauvegarde des artefacts (modèle, scaler, encodeur, features)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "encoder = LabelEncoder()\n",
    "encoder.fit([\"Sain\", \"À risque\"])\n",
    "save_comportemental(model, scaler, encoder, feature_names)\n",
    "logger.info(\"=== ENTRAÎNEMENT TERMINÉ ===\")"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3 (ipykernel)",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.10.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 2
}

NameError: name 'null' is not defined

In [4]:
# Ajout du répertoire racine au PYTHONPATH
import sys
from pathlib import Path

# On suppose que le notebook est dans models/comportemental/
# La racine du projet est deux niveaux au-dessus
root_dir = Path.cwd().parent.parent
sys.path.insert(0, str(root_dir))